# Optimal sailboat foils using polynomials
The purpose of the utility is to produce **3D-printable templates** from specified foil sections, to help sailboat builders accurately shape foils (see the discussion by Australian boat designer [Michael Storer](https://www.storerboatplans.com/boat-design/performance/naca-and-highly-accurate-centreboard-and-rudder-sections-is-it-possible/) and other online resources).

DTIC_ADA189047).
These foil shapes are specified as described in that article using polynomials for the leading and trailing edge shapes, between which are flat sections. 

This utility implements a correction to the notation for the trailing edge shape in the original Pollock paper, as suggested by Michael Storer.

This utility constructs flat-sided sections for daggerboards, rudders and other sailboat foils calculated to be optimal by  [Neil Pollock](https://archive.org/details/The utility uses [ezdxf](https://github.com/mozman/ezdxf) to write 2D dxf files,
and [CadQuery](https://github.com/CadQuery/cadquery) to generate 3D stl files from the 2D dxf files.


## How to use this utility
The simplest way to run this utility is in the cloud on a free open-source web service called [Binder](https://mybinder.org/, by: 
1. Launch the utility in the cloud by clicking this button:

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/seastate/CQFoil/HEAD?urlpath=%2Fdoc%2Ftree%2FcqfoilP.ipynb)

    Note: If the utility has not been run recently, there may be a delay in launching it while Binder rebuilds its environment. Relaunching will be much faster.

2. When the page opens, select `Run All Cells` under the `Run` menu.

3. After some initialization output, the notebook will present textboxes in which foil parameters can be entered.
   - To update a parameter, type the new value into the corresponding textbox and press `enter`.
   - After each parameter update, the foil and template configurations will be plotted as cross-sections so you can verify the foil shape.
   - 2-dimensional CAD renderings of the current foil and corresponding template are also saved, as `DXF` files.

4. When you have the foil section you want, click the `Create template` button to generate a 3D-printable template file, as an `STL` file.

5. When you have the `STL` file you want, `right-click` on that file in the file list at left and select `Download` to download it onto your computer, ready to send to a 3D printer.

### Additional information
This page is a web document called a [Jupyter notebook](https://jupyter.org/).
Jupyter notebooks have "code cells" containing [Python](https://www.python.org/) code.
You can see the code itself by clicking on the blue bar at the left.
> You can execute a single code cell by clicking on it and pressing `shift-enter`, or by selecting `Run cell` under the `Run` menu.
> Alternatively (and usually easier) you can set up the utility all at once by selecting `Run all cells` under the `Run` menu.

The utility has three section, executed in separate code cells:
1. Loading `CADquery` and other packages
2. Specifying parameters for the foil section and template
3. Exporting a 3D-dimensional file for 3D-printing


### Load `CADquery` and other packages

In [ ]:
import cadquery as cq
from math import *
import ipywidgets as widgets
import matplotlib.pyplot as plt
#plt.ion()
from foil_poly import foil_half_edgeP as fheP

### Set up textboxes to specify the foil and template sections.

In [ ]:
# Create textboxes for parameter input, setting default starting foil parameters
global g, clip, extrude, name, offset
global fig1, fig2
#fig1 = plt.figure(figsize=(7,2))
#fig2 = plt.figure(figsize=(7,2))

_chord=widgets.FloatText(value=280,width=10,description = r"$c$")
_Tfrac=widgets.FloatText(value=0.0786,description = r"$T_{frac}$")
_LEfrac=widgets.FloatText(value=0.2,description = r"$LE_{frac}$")
_TEfrac=widgets.FloatText(value=0.4,description = r"$TE_{frac}$")
_clip=widgets.FloatText(value=0,description = r"P")
_extrude=widgets.FloatText(value=20,description = r"$W$")
_name=widgets.Text(value='DemoFoilP',description = r" ")
_offset=widgets.FloatText(value=5,description = r"$\Delta$")
_thicknessTE=widgets.FloatText(value=0,description = r"$l_{TE}$")

ui0s = widgets.VBox([_chord,_Tfrac,_LEfrac,_TEfrac,_name])
ui1s = widgets.VBox([_clip,_extrude,_offset,_thicknessTE])

h = '30px'
L_chord = widgets.Label(value='chord length ($mm$)',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
L_Tfrac = widgets.Label(value='fractional thickness',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
L_LEfrac = widgets.Label(value='fractional LE length',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
L_TEfrac = widgets.Label(value='fractional TE length',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
L_clip = widgets.Label(value='clip position ($mm$)',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
L_extrude = widgets.Label(value='extrusion width ($mm$)',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
L_name = widgets.Label(value='foil name',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
L_offset = widgets.Label(value='template offset ($mm$)',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
L_thicknessTE = widgets.Label(value='TE thickness ($mm$)',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
ui0Ls = widgets.VBox([L_chord,L_Tfrac,L_LEfrac,L_TEfrac,L_name])
ui1Ls = widgets.VBox([L_clip,L_extrude,L_offset,L_thicknessTE])

ui01g = widgets.HBox([ui0Ls,ui0s,ui1s,ui1Ls])

def get_foil_P(chord_,Tfrac_,LEfrac_,TEfrac_,clip_,extrude_,name_,offset_,thicknessTE_):
    global g, clip, extrude, name, offset, thicknessTE
    global fig1, fig2
    g = fheP(chord=chord_,Tfrac=Tfrac_,LEfrac=LEfrac_,TEfrac=TEfrac_)
    g.set_pars()
    g.get_foil()
    clip = clip_
    extrude = extrude_
    name = name_
    offset = offset_
    thicknessTE = thicknessTE_
    g.plot()
    g.ax.set_title(name+' outline')
    # Write dxf files for 2D renditions of the foil and template
    g.dxfwrite(filename=name+'.dxf',scale=1)
    # Get 2D template with foil cutout (female)
    g.get_foil(template=True,offset=offset,thicknessTE=thicknessTE)
    g.dxfwrite(filename=name+'_template.dxf',scale=1,clip=clip)
    g.plot()
    g.ax.set_title(name+' template')
    g.fig.canvas.draw()
    g.fig.canvas.flush_events()
    plt.pause(0.25)

outg = widgets.interactive_output(get_foil_P,{'chord_':_chord,
                                                'Tfrac_':_Tfrac,
                                                'LEfrac_':_LEfrac,
                                                'TEfrac_':_TEfrac,
                                                'clip_':_clip,
                                                'extrude_':_extrude,
                                                'name_':_name,
                                                'offset_':_offset,
                                                'thicknessTE_':_thicknessTE})

display(ui01g,outg)

## Parameters

> Note: The default parameters correspond approximately to the optimal section by Pollock (1987) for a foil in which the thickness is 8% of the chord.

Parameters for foil size and shape are as specified by [Saporito *et al*. (2020)](https://research.chalmers.se/publication/519549/file/519549_Fulltext.pdf):
- Chord length ($c$) is specified in milimeters
- Foil thickness ($T_{frac}$), leading edge length ($LE_{frac}$) and trailing edge length ($TE_{frac}$) are specified as fractions of chord length (*e.g.* a chord length of $c=100mm$ and a fractional foil thickness $T_{frac}=0.1$ means the foil is $10mm$ thick)

Several other parameters are used to generate a template from the foil:
- The template offset ($\Delta$) is the thickness of the template above, in front of and behind the foil
- The extrusion width ($W$) is the width of the template (that is, the length in the $Z$-direction if the foil outline lies in the $XY$ plane)
- The clip position ($P$) is used if separate templates are wanted for the leading and trailing edges
    - If $P=0$, the template covers the whole foil
    - If $P>0$, the template covers only the leading P millimeters of the foil (*e.g.* $P=100$ results in a template for the forward $100mm$ of the foil)
    - If $P<0$, the template covers only the trailing -P millimeters of the foil (*e.g.* $P=-100$ results in a template for the aft $100mm$ of the foil)
- The trailing edge thickness ($l_{TE}$) is used if a squared-off trailing edge is wanted (*e.g.*, $l_{TE}=0$ results in a sharp trailing edge, $l_{TE}=2$ results in a trailing edge with $2mm$ thickness, *etc*.)


### Set up a button to export a 3D-printable template file.
- Each button press creates a 3D-rendering of the template and an STL file corresponding to that shape.
- Subsequent button presses create a new 3D-rendering **below** the previous one.
- To remove an inconveniently long stack of images, select the code block and press `shift-enter` to execute it.

> Note: You may need to enlarge the graphics window (tab at lower right) if the rendering is not immediately visible.

### Downloading the STL file
- To make the file list visible (if it isn't already) click on the folder icon at the top left of the page
- Use your mouse to rotate and zoom the image
- Right-click on a file and select `Download` to save the file to your computer.

In [ ]:
button7 = widgets.Button(description="Create template")
output7 = widgets.Output()

buttonsV = widgets.VBox([button7, output7])
display(buttonsV)

@output7.capture()
def on_button_clicked7(b):
    global g, clip, extrude, name, offset, thicknessTE
    global fig1, fig2
    # Write dxf files for 2D renditions of the foil and template, then make stl files
    # for 3D renditions of the template.
    #g.dxfwrite(filename=name+'.dxf',scale=1)
    # Get 2D template with foil cutout (female)
    #g.get_foil(template=True,offset=offset,thicknessTE=thicknessTE)
    #g.dxfwrite(filename=name+'_template.dxf',scale=1,clip=clip)
    # clear output to prepare for interative adjustments
    #g.plot()
    #g.ax.set_title(name+' template')
    #g.fig.canvas.draw()
    #g.fig.canvas.flush_events()
    #plt.pause(0.25)
    G = cq.importers.importDXF(filename=name+'_template.dxf')
    G.extrude(20).export(name+'_template.stl')
    display(G.extrude(20))

button7.on_click(on_button_clicked7)